In [1]:
import os

In [2]:
%pwd

'/workspaces/MLOps-Car-Price-Prediction-System/notebooks'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/workspaces/MLOps-Car-Price-Prediction-System'

In [6]:
import pandas as pd

In [7]:
data = pd.read_csv("artifacts/data_ingestion/car_data.csv")

data.head()

,Car_Name,Year,Selling_Price,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner
0,ritz,2014,3.35,5.59,27000,Petrol,Dealer,Manual,0
1,sx4,2013,4.75,9.54,43000,Diesel,Dealer,Manual,0
2,ciaz,2017,7.25,9.85,6900,Petrol,Dealer,Manual,0
3,wagon r,2011,2.85,4.15,5200,Petrol,Dealer,Manual,0
4,swift,2014,4.60,6.87,42450,Diesel,Dealer,Manual,0


In [8]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 301 entries, 0 to 300
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Car_Name       301 non-null    str    
 1   Year           301 non-null    int64  
 2   Selling_Price  301 non-null    float64
 3   Present_Price  301 non-null    float64
 4   Kms_Driven     301 non-null    int64  
 5   Fuel_Type      301 non-null    str    
 6   Seller_Type    301 non-null    str    
 7   Transmission   301 non-null    str    
 8   Owner          301 non-null    int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 21.3 KB


In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    data_file: str
    status_file: str
    all_schema: dict

In [9]:
from Car_Price_Prediction_System.constants import *
from Car_Price_Prediction_System.utils.common import read_yaml, create_directories

In [ ]:
class ConfigurationManager:
    def __init__(self, config_filepath=Config_Filepath, params_filepath=Params_Filepath, schema_filepath=Schema_Filepath):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        schema = self.schema.COLUMNS

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir = config.root_dir,
            data_file = config.data_file,
            status_file = config.status_file,
            all_schema = schema
        )

        return data_validation_config

In [11]:
from Car_Price_Prediction_System import logger

In [19]:
class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config
    
    def validate_all_columns(self) -> bool:
        try:
            data = pd.read_csv(self.config.data_file)
            csv_columns = set(data.columns)
            schema_columns = set(self.config.all_schema)
            validation_status = csv_columns == schema_columns

            with open(self.config.status_file, "w") as file:
                file.write(f"Validation Status: {validation_status}")

            logger.info(f"Validation Status Saved To: {self.config.status_file}")
        except Exception as e:
            raise e

In [20]:
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config=data_validation_config)
    data_validation.validate_all_columns()
except Exception as e:
    raise e

[2026-07-21 14:40:37,925] : INFO : common : Created File: <_io.TextIOWrapper name='config/config.yaml' mode='r' encoding='UTF-8'>
[2026-07-21 14:40:37,928] : INFO : common : Created File: <_io.TextIOWrapper name='config/params.yaml' mode='r' encoding='UTF-8'>
[2026-07-21 14:40:37,931] : INFO : common : Created File: <_io.TextIOWrapper name='config/schema.yaml' mode='r' encoding='UTF-8'>
[2026-07-21 14:40:37,932] : INFO : common : Created Directory: artifacts/data_validation
[2026-07-21 14:40:37,938] : INFO : 780176728 : Validation Status Saved To: artifacts/data_validation/status.txt
